#### Massive attacks

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [3]:
base = external_data

In [4]:
my_regions = [
    'Ukraine', 'Kyiv oblast', 'west', 'Khmelnytskyi oblast',
    'Kyiv', 'Lviv oblast', 'Starokostiantyniv',
    'Kyiv and Starokostiantyniv',
    'Kyiv oblast and Khmelnytskyi oblast', 'center'
]

In [5]:
df_daily = pd.read_csv(base / 'missile_attacks_daily.csv', low_memory=False)

df_weapons = pd.read_csv(base / 'missiles_and_uavs.csv')

df = df_daily.merge(df_weapons[['model', 'category']], how='left', on='model')

print(f'Rows after merge: {len(df)}')
print(f'Categories: {df["category"].value_counts().to_dict()}')
display(df[['time_start', 'model', 'category', 'launched', 'target']].head(5))


Rows after merge: 3330
Categories: {'UAV': 2140, 'cruise missile': 657, 'ballistic missile': 397, 'surface-to-air missile': 112, 'surface-to-air and ballistic': 10, 'guided bomb': 5}


,time_start,model,category,launched,target
0,2026-01-24 18:30,Shahed-136/131,UAV,102.0,Ukraine
1,2026-01-24 18:30,C-300 and Iskander-M,surface-to-air and ballistic,2.0,Ukraine
2,2026-01-23 18:00,X-59/X-69,cruise missile,1.0,Ukraine
3,2026-01-23 18:00,X-22 and X-32,cruise missile,12.0,Kyiv oblast
4,2026-01-23 18:00,Shahed-136/131,UAV,375.0,Ukraine


#### Filter by the region

In [6]:
df = df[df['target'].isin(my_regions)]
df['date_start'] = pd.to_datetime(df['time_start'], format='mixed').dt.date.astype('datetime64[ns]')

#### Pivot Table

In [7]:
pivot = df.pivot_table(
    index='date_start',
    columns='category',
    values='launched',
    aggfunc='sum'
)
pivot['Total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('Total', ascending=False)

print(f'Days with attacks: {len(pivot)}')
display(pivot.fillna('').head(10))


Days with attacks: 761


category,UAV,ballistic missile,cruise missile,guided bomb,surface-to-air and ballistic,surface-to-air missile,Total
date_start,,,,,,,
2025-09-06,810.0,4.0,9.0,,,,823.0
2025-07-08,782.0,6.0,7.0,,,4.0,799.0
2025-12-05,653.0,14.0,34.0,,,,701.0
2025-10-29,653.0,,,,,,653.0
2025-12-22,635.0,,,,,,635.0
2025-11-28,596.0,9.0,27.0,,,,632.0
2025-08-20,574.0,2.0,34.0,,,,610.0
2025-08-27,598.0,,,,,,598.0
2025-07-11,597.0,,,,,,597.0


In [8]:
out = pivot.reset_index()
out.to_csv(base / 'massive_attacks_by_category.csv', index=False)